### Data Parser Notebook

This notebook contains code to parse daily stock data from raw price data into change over time data for training. It reads from CSV formatted raw close and volume tables to generate multiple different kinds of data points.

In [ ]:
from datetime import date
import pandas as pd
from pandas import DataFrame
import numpy as np
import math
import talib as ta

DATE_LEN = 10
DATA_POINTS_PER_TICKER = 200
CSV_HEADER_OFFSET = 2
MAX_RAND = 4

In [ ]:
# Date parsing for standardized access with data frames
def parse_date(date_str:str) -> date:
    items = date_str.split('-')[:DATE_LEN]

    return date(year=int(items[0]), month=int(items[1]), day=int(items[2]))

def format_date(date: date) -> str:
    year, month, day = date.year, date.month, date.day

    return f"{year}-{month}-{day}"

In [ ]:
# Function to generate a data point with mean centered prices (No SMA or RSI)
def generate_price_centered_point(df: DataFrame, price_col: str, volume_col: str, dates: DataFrame, label_index: int, offset_len: int = 126, train_len: int = 252, log = True) -> list[float]:
    fend = label_index - offset_len
    fstart  = fend - train_len

    # One extra day for prices to calculate initial gain, fill NaNs forward
    raw_prices = df[price_col][fstart:fend].astype('float64')

    # Mean center the prices
    raw_std = np.std(raw_prices) + 1e-5
    raw_mean = np.mean(raw_prices)

    if log: print("Raw prices:", raw_prices)

    feature_prices = (raw_prices - raw_mean) / raw_std

    if log: print("Feature Prices:", feature_prices)

    # Calculate the label change and mean center it
    fprice_end = raw_prices.tail(1).iloc[0]
    fprice_start = raw_prices.head(1).iloc[0]

    feature_change = (fprice_end - fprice_start) / fprice_start
    
    label_price = (float(df[price_col][label_index]) - raw_mean) / raw_std

    if log: print("Label Price:", label_price)

    return list(feature_prices) +  [ feature_change, label_price ]

In [ ]:
# Function to generate a data point with mean centered SMA and non-centered RSI
def generate_sma_point(df: DataFrame, price_col: str, volume_col: str, dates: DataFrame, label_index: int, offset_len: int = 52, train_len: int = 252, window: int = 14, log = True, context: DataFrame = None) -> list[float]:
    # Calculate feature start and end based on offset and train lens
    fend = label_index - offset_len
    fstart  = fend - train_len

    # Get the rolling window for the SMA, must be window indices before fstart to prevent NaN entries
    price_window = df[price_col][(fstart-window):fend].astype('float64').rolling(window=window)

    if log: print("14 day Window:", price_window)

    # Get the train_len SMA
    raw_sma = price_window.mean().tail(train_len)

    if log: print("Raw SMA 14:", raw_sma)

    # Mean center the SMA
    raw_mean = raw_sma.mean()
    raw_std = raw_sma.std() + 1e-8

    feature_sma = (raw_sma - raw_mean) / raw_std

    if log: print("Feature SMA:", feature_sma)

    # Get 14 day rolling changes to calculate RSI
    feature_rsi = ta.RSI(df[price_col][(fstart-window-1):fend].astype('float64'), timeperiod=window).tail(train_len)
    
    if log: print("Feature TSI:", feature_rsi)

    # Get the mean centered label price
    label_price = (float(df[price_col][label_index]) - raw_mean) / raw_std

    if log: print("Label Price:", label_price)

    # Assert the feature_sma and feature_rsi are both the desired length
    assert(len(feature_sma) == len(feature_rsi) == train_len)

    return list(feature_sma) + list(feature_rsi) + [label_price]


In [ ]:
# Function to generate many data points with a given generator function
def generate_data_points(df: DataFrame, dates: DataFrame, price_col: str, volume_col: str, amount: int, window:int = 0,context: DataFrame = None, offset_len: int = 126, log: bool = True, generator=generate_price_centered_point) -> list[list[float]]:    
    # Fill NaN values forward (as no-trade = same price) to prevent NaNs from polluting the data set
    price_data = df[price_col][CSV_HEADER_OFFSET:].ffill()

    # Find the first and last valid dates for labels
    start_valid = price_data.first_valid_index() + window + 1
    end_valid = price_data.last_valid_index()

    if log: print("Valid Date Range:", df['Price'][start_valid], "|", df['Price'][end_valid])

    # Exlude set to prevent duplicates
    exclude = set({})

    output = []

    for _ in range(amount):
        # Label must have offset and one year valid before it
        start_range = start_valid + offset_len + 252 + 1

        if start_range >= end_valid:
            if log: print("No range available for training, returning empty")
            return []

        # Try to generate a random label
        label_index = np.random.randint(start_range, end_valid)

        tries = 0

        while (label_index in exclude or math.isnan(float(price_data[:][label_index]))) and tries < MAX_RAND:
            label_index = np.random.randint(start_range, end_valid)
            tries += 1

            if tries == MAX_RAND:
                if log: print("Random exceeded max tries, returning early")
                return output
        
        # Add index to exclude
        exclude.add(label_index)

        # Call generator function differently depending on if context is defined
        if not(context is None):
            point = generator(df, price_col, volume_col, dates, label_index, context=context, offset_len=offset_len, log=log)
        else:
            point = generator(df, price_col, volume_col, dates, label_index, window=window, offset_len=offset_len, log=log)

        if log: print("Point:", point)

        # Add the point if it has no NaN values
        if np.count_nonzero(np.isnan(point)) == 0:
            output.append(point)

    return output

In [ ]:
# Load raw price info
df = pd.read_csv('raw_phist.csv')

# Note: These objects are lists of column names, not just one name, as the csv loads in with Close, Close.1, Close.2,...
dates = df.filter(axis=1, like='Price')
prices = df.filter(axis=1, like='Close')
volumes = df.filter(axis=1, like='Volume')

In [ ]:
# Specifically generate an SMA file with a given window and offset len
def generate_sma(window: int, offset_len: int):
    data = []

    log = False

    save_loc = f'stock_sma_{window}_{offset_len}_curated.csv'

    # Loop through each price and volume column and generate points using that columns info
    for (price, volume) in zip(prices, volumes):
        point_data = generate_data_points(df, dates, price, volume, DATA_POINTS_PER_TICKER, log=log, generator=generate_sma_point, window=window, offset_len=offset_len) 
        #print(point_data)

        # If NaN count is greater than 0, something has went wrong, so don't add these points
        if np.count_nonzero(np.isnan(point_data)) > 0:
            #print("Data parsing failed for:", price, volume)
            #print(point_data)
            continue

        data = data + point_data

    assert (np.count_nonzero(np.isnan(data)) == 0)

    import os

    # Overwrite and save the file
    if os.path.exists(save_loc): os.remove(save_loc)

    np.savetxt(save_loc, np.array(data).T, delimiter=',')

In [ ]:
# Generate SMA with window = 3, and offset len = 3
generate_sma(3, 3)

In [ ]:
# This cell will generate 36 data sets, so watch out
interval_list = [3,7,15,31,63,126]

for window in interval_list:
    for offset_len in interval_list:
        generate_sma(window, offset_len)